# 04 · Gold: star schema
Line-grain fact (invoices +, credit memos −), conformed dimensions with deterministic hashed surrogate keys, an as-of join to the SCD2 customer dimension, Unknown members (-1) so no revenue is dropped, and a daily reconciliation table against Business Central.

Gold is rebuilt each run, so late-arriving facts are always re-mapped through the as-of join. If Gold is made incremental, add a lookback window (re-merge the last N days).

In [ ]:
%%sql
CREATE OR REPLACE TABLE gold_dim_date AS
SELECT CAST(date_format(d, 'yyyyMMdd') AS INT) AS date_key, d AS full_date,
       year(d) AS year, quarter(d) AS quarter, month(d) AS month,
       date_format(d, 'MMMM') AS month_name, date_format(d, 'EEEE') AS day_name,
       dayofweek(d) IN (1, 7) AS is_weekend
FROM (SELECT explode(sequence(DATE'2025-01-01', DATE'2026-12-31', INTERVAL 1 DAY)) AS d)

In [ ]:
%%sql
CREATE OR REPLACE TABLE gold_dim_customer AS
SELECT xxhash64(Company, No, valid_from) AS customer_sk,
       Company, No AS customer_no, COALESCE(Name, 'Unknown name') AS customer_name,
       Customer_Type, Region_Code, City, State, valid_from, valid_to, is_current
FROM silver_customer
UNION ALL
SELECT -1, 'UNKNOWN', 'UNKNOWN', 'Unknown customer', 'UNKNOWN', 'UNKNOWN', 'UNKNOWN', 'UNKNOWN',
       TIMESTAMP'1900-01-01', TIMESTAMP'9999-12-31', true

In [ ]:
%%sql
CREATE OR REPLACE TABLE gold_dim_product AS
SELECT xxhash64(Company, No) AS product_sk, Company, No AS item_no,
       Description AS product_name, Item_Category_Code AS category, GTIN, GST_Group,
       Unit_Price AS current_unit_price
FROM silver_item
UNION ALL
SELECT -1, 'UNKNOWN', 'UNKNOWN', 'Unknown product', 'UNKNOWN', NULL, 'UNKNOWN', NULL

In [ ]:
%%sql
CREATE OR REPLACE TABLE gold_dim_location AS
SELECT xxhash64(Company, Code) AS location_sk, Company, Code AS location_code,
       Name AS location_name, Location_Type, City, State
FROM silver_location
UNION ALL
SELECT -1, 'UNKNOWN', 'UNKNOWN', 'Unknown location', 'UNKNOWN', 'UNKNOWN', 'UNKNOWN'

In [ ]:
%%sql
CREATE OR REPLACE TABLE gold_fact_sales_line AS
WITH lines AS (
  SELECT 'INVOICE' AS document_type, h.Company, h.No AS document_no, l.Line_No AS line_no,
         h.Posting_Date, h.Sell_to_Customer_No AS customer_no, l.Item_No, l.Location_Code, h.Business_Unit,
         l.Quantity AS quantity, l.Unit_Price AS unit_price,
         l.Amount AS amount_excl_gst, l.Amount_Incl_GST AS amount_incl_gst
  FROM silver_sales_invoice_header h
  JOIN silver_sales_invoice_line l ON l.Company = h.Company AND l.Document_No = h.No
  UNION ALL
  SELECT 'CREDIT_MEMO', h.Company, h.No, l.Line_No,
         h.Posting_Date, h.Sell_to_Customer_No, l.Item_No, l.Location_Code, h.Business_Unit,
         -l.Quantity, l.Unit_Price, -l.Amount, -l.Amount_Incl_GST
  FROM silver_sales_cr_memo_header h
  JOIN silver_sales_cr_memo_line l ON l.Company = h.Company AND l.Document_No = h.No
)
SELECT CAST(date_format(x.Posting_Date, 'yyyyMMdd') AS INT) AS date_key,
       COALESCE(c.customer_sk, -1)  AS customer_sk,
       COALESCE(p.product_sk,  -1)  AS product_sk,
       COALESCE(lo.location_sk, -1) AS location_sk,
       x.*
FROM lines x
LEFT JOIN gold_dim_customer c
       ON  c.Company = x.Company AND c.customer_no = x.customer_no
       AND CAST(x.Posting_Date AS TIMESTAMP) >= c.valid_from
       AND CAST(x.Posting_Date AS TIMESTAMP) <  c.valid_to          -- as-of join
LEFT JOIN gold_dim_product p   ON p.Company = x.Company AND p.item_no = x.Item_No
LEFT JOIN gold_dim_location lo ON lo.Company = x.Company AND lo.location_code = x.Location_Code

In [ ]:
%%sql
-- expect: 6645 rows, 1 unknown product (discontinued item still counted), 0 unknown customers
SELECT COUNT(*) AS fact_rows,
       SUM(CASE WHEN product_sk  = -1 THEN 1 ELSE 0 END) AS unknown_products,
       SUM(CASE WHEN customer_sk = -1 THEN 1 ELSE 0 END) AS unknown_customers,
       ROUND(SUM(amount_excl_gst), 2) AS total_net_sales
FROM gold_fact_sales_line

In [ ]:
(spark.read.option("header", True).csv("Files/reference/bc_control_totals.csv")
     .write.mode("overwrite").format("delta").saveAsTable("ref_bc_control_totals"))

In [ ]:
%%sql
CREATE OR REPLACE TABLE gold_recon_daily_sales AS
SELECT b.Company,
       CAST(b.Posting_Date AS DATE)                    AS posting_date,
       CAST(b.Net_Sales_Excl_GST AS DECIMAL(18,2))     AS bc_net_sales,
       COALESCE(f.net, 0)                              AS gold_net_sales,
       COALESCE(f.net, 0) - CAST(b.Net_Sales_Excl_GST AS DECIMAL(18,2)) AS difference
FROM ref_bc_control_totals b
LEFT JOIN (SELECT Company, Posting_Date, SUM(amount_excl_gst) AS net
           FROM gold_fact_sales_line GROUP BY Company, Posting_Date) f
  ON f.Company = b.Company AND f.Posting_Date = CAST(b.Posting_Date AS DATE)

In [ ]:
%%sql
-- expect days_wrong = 0
SELECT COUNT(*) AS days, SUM(CASE WHEN ABS(difference) > 0.01 THEN 1 ELSE 0 END) AS days_wrong
FROM gold_recon_daily_sales